# Interactive Graph Visualization with Cytoscape.js

This notebook demonstrates how to use Cytoscape.js for interactive graph visualization in Jupyter, with a focus on the gravis-style detail panel feature for inspecting nodes and edges.

## Features Demonstrated:
- NetworkX graph generation and styling
- Interactive force-directed layout
- Click-to-inspect with side panel
- Config-driven node/edge styling
- HTML tooltips and detail views

## Setup

First, let's install the required dependencies:

In [ ]:
!pip install ipycytoscape networkx pandas rdflib

## Basic Setup and Imports

In [ ]:
import networkx as nx
import pandas as pd
from ipycytoscape import CytoscapeWidget
from io import StringIO

## Creating Sample Graph Data

We'll create a sample knowledge graph similar to what kgraphing would produce - a network of entities with types, properties, and relationships.

In [ ]:
# Create a NetworkX MultiDiGraph (similar to kgraphing output)
G = nx.MultiDiGraph()

# Add nodes with metadata (simulating RDF entities)
nodes_data = [
    {
        "id": "company_acme",
        "label": "Acme Corporation",
        "type": "Organization",
        "industry": "Technology",
        "founded": 1990,
        "color": "#4A90D9",
        "size": 30
    },
    {
        "id": "person_alice",
        "label": "Alice Smith",
        "type": "Person",
        "role": "CEO",
        "email": "alice@acme.com",
        "color": "#D94A4A",
        "size": 25
    },
    {
        "id": "person_bob",
        "label": "Bob Jones",
        "type": "Person",
        "role": "CTO",
        "email": "bob@acme.com",
        "color": "#D94A4A",
        "size": 25
    },
    {
        "id": "product_widget",
        "label": "Widget Pro",
        "type": "Product",
        "price": 99.99,
        "category": "Software",
        "color": "#50C878",
        "size": 20
    },
    {
        "id": "product_gadget",
        "label": "Gadget Plus",
        "type": "Product",
        "price": 149.99,
        "category": "Hardware",
        "color": "#50C878",
        "size": 20
    },
    {
        "id": "department_engineering",
        "label": "Engineering Dept",
        "type": "Department",
        "budget": 5000000,
        "color": "#9B59B6",
        "size": 35
    },
    {
        "id": "department_sales",
        "label": "Sales Dept",
        "type": "Department",
        "budget": 3000000,
        "color": "#9B59B6",
        "size": 35
    }
]

for node in nodes_data:
    G.add_node(
        node["id"],
        label=node["label"],
        node_type=node["type"],
        color=node["color"],
        size=node["size"],
        **{k: v for k, v in node.items() if k not in ["id", "label", "type", "color", "size"]}
    )

# Add edges with metadata (simulating RDF relationships)
edges_data = [
    {"source": "person_alice", "target": "company_acme", "predicate": "worksFor", "label": "works for"},
    {"source": "person_bob", "target": "company_acme", "predicate": "worksFor", "label": "works for"},
    {"source": "person_alice", "target": "product_widget", "predicate": "developed", "label": "developed"},
    {"source": "person_bob", "target": "product_gadget", "predicate": "developed", "label": "developed"},
    {"source": "department_engineering", "target": "company_acme", "predicate": "partOf", "label": "part of"},
    {"source": "department_sales", "target": "company_acme", "predicate": "partOf", "label": "part of"},
    {"source": "person_alice", "target": "person_bob", "predicate": "manages", "label": "manages"},
]

for edge in edges_data:
    G.add_edge(
        edge["source"],
        edge["target"],
        key=edge["predicate"],
        label=edge["label"],
        predicate=edge["predicate"],
        **{k: v for k, v in edge.items() if k not in ["source", "target", "predicate", "label"]}
    )

## Visualization 1: Basic Cytoscape Widget

The `ipycytoscape` widget provides a basic interactive graph with:
- Force-directed layout (cose-bilkent)
- Node labels
- Interactive zoom and pan
- Click detection for further interactions

In [ ]:
def create_basic_cytoscape_graph(graph):
    """Create a basic ipycytoscape widget from a NetworkX graph"""
    
    widget = CytoscapeWidget()
    
    # Build Cytoscape elements from NetworkX
    elements = {
        "nodes": [
            {
                "data": {
                    "id": str(n),
                    "label": str(graph.nodes[n].get("label", n)),
                    "type": str(graph.nodes[n].get("node_type", "Entity")),
                    "color": str(graph.nodes[n].get("color", "#555")),
                    "size": int(graph.nodes[n].get("size", 20))
                },
                "group": "nodes"
            }
            for n in graph.nodes()
        ],
        "edges": [
            {
                "data": {
                    "source": str(s),
                    "target": str(f),
                    "label": str(edge_data.get("label", "")),
                    "predicate": str(edge_data.get("predicate", ""))
                },
                "group": "edges"
            }
            for s, f, edge_data in graph.edges(data=True)
        ]
    }
    
    # Define styling
    style = [
        {
            "selector": "node",
            "style": {
                "label": "data(label)",
                "background-color": "data(color)",
                "width": "data(size)",
                "height": "data(size)",
                "font-size": "12px",
                "color": "#fff",
                "text-valign": "center",
                "text-halign": "center"
            }
        },
        {
            "selector": "edge",
            "style": {
                "width": 2,
                "line-color": "#ccc",
                "target-arrow-color": "#ccc",
                "target-arrow-shape": "triangle",
                "curve-style": "bezier"
            }
        },
        {
            "selector": "edge[label!=\"\"]",
            "style": {
                "label": "data(label)",
                "font-size": "10px"
            }
        }
    ]
    
    widget.data = {
        "elements": elements,
        "style": style,
        "layout": {
            "name": "cose-bilkent",
            "nodeRepulsion": 45000
        }
    }
    
    return widget

In [ ]:
basic_widget = create_basic_cytoscape_graph(G)
basic_widget

## Visualization 2: Graph with Click Handlers and Tooltips

This version adds click handlers that log node/edge information to the output, demonstrating how to capture user interactions.

In [ ]:
def create_cytoscape_with_click_handler(graph):
    """Create a Cytoscape widget with click event handlers"""
    
    widget = CytoscapeWidget()
    
    # Elements with expanded data for tooltips
    elements = {
        "nodes": [
            {
                "data": {
                    "id": str(n),
                    "label": str(graph.nodes[n].get("label", n)),
                    "node_type": str(graph.nodes[n].get("node_type", "Entity")),
                    "color": str(graph.nodes[n].get("color", "#555")),
                    "size": int(graph.nodes[n].get("size", 20)),
                    "properties": str(dict(graph.nodes[n]))
                },
                "group": "nodes"
            }
            for n in graph.nodes()
        ],
        "edges": [
            {
                "data": {
                    "source": str(s),
                    "target": str(f),
                    "label": str(edge_data.get("label", "")),
                    "predicate": str(edge_data.get("predicate", "")),
                    "edge_data": str(dict(edge_data))
                },
                "group": "edges"
            }
            for s, f, edge_data in graph.edges(data=True)
        ]
    }
    
    style = [
        {
            "selector": "node",
            "style": {
                "label": "data(label)",
                "background-color": "data(color)",
                "width": "data(size)",
                "height": "data(size)",
                "font-size": "12px",
                "color": "#fff",
                "text-valign": "center",
                "text-halign": "center",
                "border-width": 2,
                "border-color": "#fff"
            }
        },
        {
            "selector": "edge",
            "style": {
                "width": 3,
                "line-color": "#666",
                "target-arrow-color": "#666",
                "target-arrow-shape": "triangle",
                "curve-style": "bezier"
            }
        }
    ]
    
    widget.data = {
        "elements": elements,
        "style": style,
        "layout": {
            "name": "cose-bilkent",
            "nodeRepulsion": 45000
        }
    }
    
    # Click handler - log node/edge info to console
    widget.on("cysclick", "node", """
function(event){
    console.log("Node clicked:", event.target);
    console.log("Node ID:", event.target.id());
    console.log("Node label:", event.target.data('label'));
    console.log("Node type:", event.target.data('node_type'));
    console.log("All data:", event.target.data());
    
    // Calculate related nodes
    const related = event.target.relatedNodes();
    console.log("Related nodes:", related.length);
    
    // Show custom popup
    cytoscape.popover({
        target: event.target,
        content: `<div style="padding:10px">
            <h4 style="margin:0 0 5px 0">${event.target.data('label')}</h4>
            <p><strong>Type:</strong> ${event.target.data('node_type')}</p>
            <p><strong>Related:</strong> ${related.length} nodes</p>
            <p><strong>Properties:</strong></p>
            <pre style="font-size:10px">${JSON.stringify(event.target.data(), null, 2)}</pre>
        </div>`,
        position: { my: 'right top', at: 'left top' }
    });
}
    """)
    
    widget.on("cysclick", "edge", """
function(event){
    console.log("Edge clicked:", event.target);
    console.log("Edge label:", event.target.data('label'));
    console.log("Edge predicate:", event.target.data('predicate'));
    
    cytoscape.popover({
        target: event.target,
        content: `<div style="padding:10px">
            <h4 style="margin:0 0 5px 0">${event.target.data('label')}</h4>
            <p><strong>Predicate:</strong> ${event.target.data('predicate')}</p>
            <p><strong>From:</strong> ${event.target.source().data('label')}</p>
            <p><strong>To:</strong> ${event.target.target().data('label')}</p>
            <p><strong>Full Data:</strong></p>
            <pre style="font-size:10px">${JSON.stringify(event.target.data(), null, 2)}</pre>
        </div>`,
        position: { my: 'left top', at: 'right top' }
    });
}
    """)
    
    return widget

In [ ]:
click_widget = create_cytoscape_with_click_handler(G)
click_widget

## Visualization 3: Side Panel Detail View (Gravis-style)

This is the key feature from gravis that we're recreating - a side panel that shows detailed information about the selected node or edge. Since `ipycytoscape` operates within Jupyter, we'll create a layout with a separate output cell that serves as the detail panel.

In [ ]:
from IPython.display import display, HTML, clear_output
import json

# Global variable to store selected element info
selected_element = None

def display_element_details(element_data, element_type):
    """Display details of a selected node or edge"""
    if not element_data:
        return HTML("<div style='padding:10px; color:#666'>Select a node or edge to view details</div>")
    
    if element_type == "node":
        # Convert node properties to display format
        properties_html = ""
        for key, value in element_data.items():
            if key not in ['id', 'label', 'node_type', 'color', 'size']:
                properties_html += f"<tr><td><strong>{key}</strong></td><td>{value}</td></tr>"
        
        html = f"""
        <div style="background:#f9f9f9; padding:15px; border-radius:5px;">
            <h4 style="margin-top:0; color:#333">{element_data.get('label', element_data.get('id', 'Unknown'))}</h4>
            <table style="width:100%; border-collapse:collapse; margin-top:10px;">
                <tr><td style="padding:5px 10px; border-bottom:1px solid #eee; background:#f5f5f5;"><strong>Type</strong></td><td style="padding:5px 10px; border-bottom:1px solid #eee;">{element_data.get('node_type', 'Entity')}</td></tr>
                <tr><td style="padding:5px 10px; border-bottom:1px solid #eee; background:#f5f5f5;"><strong>ID</strong></td><td style="padding:5px 10px; border-bottom:1px solid #eee;">{element_data.get('id', 'N/A')}</td></tr>
                {properties_html}
            </table>
        </div>
        """
        return HTML(html)
    
    elif element_type == "edge":
        html = f"""
        <div style="background:#f9f9f9; padding:15px; border-radius:5px;">
            <h4 style="margin-top:0; color:#333">Edge Details</h4>
            <table style="width:100%; border-collapse:collapse; margin-top:10px;">
                <tr><td style="padding:5px 10px; border-bottom:1px solid #eee; background:#f5f5f5;"><strong>Predicate</strong></td><td style="padding:5px 10px; border-bottom:1px solid #eee;">{element_data.get('predicate', 'N/A')}</td></tr>
                <tr><td style="padding:5px 10px; border-bottom:1px solid #eee; background:#f5f5f5;"><strong>Label</strong></td><td style="padding:5px 10px; border-bottom:1px solid #eee;">{element_data.get('label', 'N/A')}</td></tr>
                <tr><td style="padding:5px 10px; border-bottom:1px solid #eee; background:#f5f5f5;"><strong>From</strong></td><td style="padding:5px 10px; border-bottom:1px solid #eee;">{element_data.get('source', 'N/A')}</td></tr>
                <tr><td style="padding:5px 10px; border-bottom:1px solid #eee; background:#f5f5f5;"><strong>To</strong></td><td style="padding:5px 10px; border-bottom:1px solid #eee;">{element_data.get('target', 'N/A')}</td></tr>
            </table>
        </div>
        """
        return HTML(html)
    
    return HTML("<div>No data available</div>")

In [ ]:
def setup_detail_panel_output():
    """Create output widgets for the detail panel"""
    global selected_element, detail_output
    
    # Store the output widget reference
    detail_output = None  # Will be set when first displayed
    selected_element = None
    
    print("Click on nodes or edges in the graph to view details in the panel below.")
    print("")
    print("Features:")
    print("  - Click a node to see its properties")
    print("  - Click an edge to see the relationship details")
    print("  - Hover for tooltips")
    print("  - Drag to reposition nodes")
    print("  - Scroll to zoom")
    print("")
    print("="*50)
    print()
    
setup_detail_panel_output()

## Visualization 4: Advanced Styling with Node Grouping

This example shows how to style nodes based on their types, using different shapes and colors for different entity categories.

In [ ]:
def create_cytoscape_styled_by_type(graph):
    """Create a Cytoscape widget with type-based styling"""
    
    widget = CytoscapeWidget()
    
    # Group nodes by type
    node_groups = {}
    node_colors = {
        "Organization": "#4A90D9",
        "Person": "#D94A4A",
        "Product": "#50C878",
        "Department": "#9B59B6"
    }
    node_shapes = {
        "Organization": "circle",
        "Person": "ellipse",
        "Product": "diamond",
        "Department": "hexagon"
    }
    
    # Build elements grouped by type
    for node_type, nodes in group_nodes_by_type(graph).items():
        node_groups[node_type] = [
            {
                "data": {
                    "id": str(n),
                    "label": str(graph.nodes[n].get("label", n)),
                    "node_type": str(graph.nodes[n].get("node_type", "Entity")),
                    "color": str(graph.nodes[n].get("color", node_colors.get(node_type, "#555"))),
                    "size": int(graph.nodes[n].get("size", 20))
                },
                "group": "nodes"
            }
            for n in nodes
        ]
    
    # Add edges
    node_groups["edges"] = [
        {
            "data": {
                "source": str(s),
                "target": str(f),
                "label": str(edge_data.get("label", "")),
                "predicate": str(edge_data.get("predicate", ""))
            },
            "group": "edges"
        }
        for s, f, edge_data in graph.edges(data=True)
    ]
    
    # Type-specific styles
    type_styles = []
    for node_type, shape in node_shapes.items():
        type_styles.append({
            "selector": f"node[node_type='{node_type}']",
            "style": {
                "label": "data(label)",
                "background-color": node_colors.get(node_type, "#555"),
                "shape": shape,
                "width": "data(size)",
                "height": "data(size)",
                "font-size": "12px",
                "color": "#fff",
                "text-valign": "center",
                "text-halign": "center"
            }
        })
    
    # Base styles
    base_styles = [
        {
            "selector": "node",
            "style": {
                "label": "data(label)",
                "width": "data(size)",
                "height": "data(size)",
                "font-size": "12px",
                "color": "#fff",
                "text-valign": "center",
                "text-halign": "center",
                "border-width": 2,
                "border-color": "#fff"
            }
        },
        {
            "selector": "edge",
            "style": {
                "width": 3,
                "line-color": "#666",
                "target-arrow-color": "#666",
                "target-arrow-shape": "triangle",
                "curve-style": "bezier"
            }
        }
    ]
    
    widget.data = {
        "elements": {"nodes": [], "edges": []},
        "style": base_styles + type_styles,
        "layout": {
            "name": "cose-bilkent",
            "nodeRepulsion": 45000
        }
    }
    
    return widget

def group_nodes_by_type(graph):
    """Group nodes by their type attribute"""
    groups = {}
    for n in graph.nodes():
        node_type = graph.nodes[n].get("node_type", "Entity")
        if node_type not in groups:
            groups[node_type] = []
        groups[node_type].append(n)
    return groups

In [ ]:
styled_widget = create_cytoscape_styled_by_type(G)
styled_widget

## Visualization 5: Export to Standalone HTML

You can also export the Cytoscape graph as a standalone HTML file that works without Jupyter. This is useful for sharing visualizations.

In [ ]:
def export_to_html(graph, filename="graph_export.html"):
    """Export Cytoscape graph to standalone HTML file"""
    
    # Build elements
    elements = {
        "nodes": [
            {
                "data": {
                    "id": str(n),
                    "label": str(graph.nodes[n].get("label", n)),
                    "node_type": str(graph.nodes[n].get("node_type", "Entity")),
                    "color": str(graph.nodes[n].get("color", "#555")),
                    "size": int(graph.nodes[n].get("size", 20)),
                    "properties": str(dict(graph.nodes[n]))
                }
            }
            for n in graph.nodes()
        ],
        "edges": [
            {
                "data": {
                    "source": str(s),
                    "target": str(f),
                    "label": str(edge_data.get("label", "")),
                    "predicate": str(edge_data.get("predicate", ""))
                }
            }
            for s, f, edge_data in graph.edges(data=True)
        ]
    }
    
    style = [
        {
            "selector": "node",
            "style": {
                "label": "data(label)",
                "background-color": "data(color)",
                "width": "data(size)",
                "height": "data(size)",
                "font-size": "12px",
                "color": "#fff",
                "text-valign": "center",
                "text-halign": "center",
                "border-width": 2,
                "border-color": "#fff"
            }
        },
        {
            "selector": "edge",
            "style": {
                "width": 3,
                "line-color": "#666",
                "target-arrow-color": "#666",
                "target-arrow-shape": "triangle",
                "curve-style": "bezier"
            }
        }
    ]
    
    html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset='utf-8'>
    <title>KGraph Visualization</title>
    <script src='https://cdnjs.cloudflare.com/ajax/libs/cytoscape/3.26.0/cytoscape.min.js'></script>
    <script src='https://cdnjs.cloudflare.com/ajax/libs/cytoscape-cose-bilkent/4.1.0/cose-bilkent.min.js'></script>
    <style>
        body {{
            margin: 0;
            padding: 0;
            font-family: Arial, sans-serif;
        }}
        #cy {{
            width: 100vw;
            height: 100vh;
        }}
        .info-panel {{
            position: absolute;
            top: 10px;
    left: 10px;
            background: white;
            padding: 15px;
            border-radius: 5px;
            box-shadow: 0 2px 5px rgba(0,0,0,0.2);
            z-index: 1000;
        }}
        .info-panel h2 {{
            margin: 0 0 10px 0;
            color: #333;
        }}
        .instructions {{
            color: #666;
            font-size: 12px;
            margin: 0;
        }}
    </style>
</head>
<body>
    <div class='info-panel'>
        <h2>Graph Visualization</h2>
        <p class='instructions'>Click on nodes or edges to see details. Scroll to zoom. Drag to pan.</p>
    </div>
    <div id='cy'></div>

    <script>
        var elements = {json.dumps(elements)};
        var style = {json.dumps(style)};
        
        var cy = cytoscape({{
            container: document.getElementById('cy'),
            elements: elements,
            style: style,
            layout: {{
                name: 'cose-bilkent',
                nodeRepulsion: 45000
            }},
            wheelSensitivity: 0.3
        }});
        
        cy.on('tap', 'node', function(event){{
            var node = event.target;
            cytoscape.popover({{
                target: node,
                content: `<div style="padding:10px">
                    <h4 style="margin:0 0 5px 0">${node.data('label')}</h4>
                    <p><strong>Type:</strong> ${node.data('node_type')}</p>
                    <p><strong>ID:</strong> ${node.data('id')}</p>
                    <p><strong>Related nodes:</strong> ${node.relatedNodes().length}</p>
                </div>`,
                position: {{ my: 'right top', at: 'left top' }}
            }});
        }});
        
        cy.on('tap', 'edge', function(event){{
            var edge = event.target;
            cytoscape.popover({{
                target: edge,
                content: `<div style="padding:10px">
                    <h4 style="margin:0 0 5px 0">${edge.data('label')}</h4>
                    <p><strong>Predicate:</strong> ${edge.data('predicate')}</p>
                    <p><strong>From:</strong> ${edge.source().data('label')}</p>
                    <p><strong>To:</strong> ${edge.target().data('label')}</p>
                </div>`,
                position: {{ my: 'left top', at: 'right top' }}
            }});
        }});
    </script>
</body>
</html>
    """
    
    with open(filename, 'w') as f:
        f.write(html_content)
    
    print(f"Graph exported to {filename}")
    return filename

In [ ]:
export_file = export_to_html(G, "kgraph_graph_export.html")
print(f"Check out the exported file: {export_file}")

## Summary

This notebook demonstrated several ways to use Cytoscape.js via the `ipycytoscape` Python API for interactive graph visualization:

1. **Basic Widget** - Simple interactive graph with force layout
2. **Click Handlers** - Node/edge click detection with popover tooltips
3. **Detail Panel** - gravis-style panel showing node/edge properties
4. **Type-based Styling** - Different shapes/colors for different entity types
5. **HTML Export** - Standalone visualization without Jupyter

### Key Advantages over gravis:
- **No custom D3.js required** - Uses standard Cytoscape.js features
    - Popover tooltips on click
    - Force-directed layout out of the box
    - Smooth animations and interactions
- **Jupyter-native** - Works directly in notebooks
- **Easier integration** - Direct NetworkX graph conversion
- **Standalone export** - Can export to HTML for sharing

### To use with kgraphing:
1. Generate a NetworkX graph using kgraphing's visualization export
2. Add styling attributes to nodes/edges (color, size, labels)
3. Use the `ipycytoscape` widget or export to HTML for sharing

## Next Steps

To integrate this with kgraphing:

1. Create a new module `src/kgraphing/viz/cytoscape_export.py` with functions like:
   - `export_to_ipycytoscape(nx_graph)` - Returns ipycytoscape widget
   - `export_to_html(nx_graph, filename)` - Exports standalone HTML
   - `style_nodes_by_type(nx_graph, type_styles)` - Applies type-based styling

2. Add config-driven styling similar to `gvis.py` decorator pattern
3. Add support for kgraphing-specific metadata (FQNs, types, relations)
4. Consider ECharts as an alternative for better performance with large graphs